In [2]:
import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [4]:
from google.colab import files

uploaded = files.upload()

Saving deliveries.csv to deliveries.csv
Saving matches.csv to matches.csv


In [45]:
matches = pd.read_csv("matches.csv")

print("Dataset Shape:", matches.shape)
matches.head()

Dataset Shape: (1095, 20)


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [46]:
team_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiant": "Rising Pune Supergiants"
}

for col in ["team1","team2","winner","toss_winner"]:
    matches[col] = matches[col].replace(team_mapping)

In [58]:
matches["venue"] = matches["venue"].replace({

    # Chinnaswamy
    "m Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "m.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",

    # HPCA
    "Himachal Pradesh Cricket Association Stadium":
        "HPCA Stadium",

    "Himachal Pradesh Cricket Association Stadium, Dharamsala":
        "HPCA Stadium",

    # Maharashtra
    "Maharashtra Cricket Association Stadium, Pune":
        "Maharashtra Cricket Association Stadium"
})

In [60]:
venue_mapping = {

    # Brabourne
    "Brabourne Stadium, Mumbai":
        "Brabourne Stadium",

    # DY Patil
    "Dr DY Patil Sports Academy":
        "DY Patil Stadium",

    "Dr DY Patil Sports Academy, Mumbai":
        "DY Patil Stadium",

    # Vizag
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam":
        "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",

    # Chinnaswamy
    "M.Chinnaswamy Stadium":
        "M Chinnaswamy Stadium",

    # Mohali
    "Punjab Cricket Association IS Bindra Stadium":
        "PCA Stadium",

    "Punjab Cricket Association IS Bindra Stadium, Mohali":
        "PCA Stadium",

    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh":
        "PCA Stadium",

    "Punjab Cricket Association Stadium, Mohali":
        "PCA Stadium",

    # Hyderabad
    "Rajiv Gandhi International Stadium":
        "Rajiv Gandhi International Stadium, Hyderabad",

    "Rajiv Gandhi International Stadium, Uppal":
        "Rajiv Gandhi International Stadium, Hyderabad",

    "Rajiv Gandhi International Stadium, Uppal, Hyderabad":
        "Rajiv Gandhi International Stadium, Hyderabad",

    # Jaipur
    "Sawai Mansingh Stadium, Jaipur":
        "Sawai Mansingh Stadium",

    # Mumbai
    "Wankhede Stadium, Mumbai":
        "Wankhede Stadium",

    # Ahmedabad
    "Narendra Modi Stadium, Ahmedabad":
        "Narendra Modi Stadium"
}

matches["venue"] = matches["venue"].replace(venue_mapping)

In [61]:
venues = sorted(matches["venue"].dropna().unique())

for v in venues:
    print(v)

Arun Jaitley Stadium
Barabati Stadium
Barsapara Cricket Stadium, Guwahati
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow
Brabourne Stadium
Buffalo Park
DY Patil Stadium
De Beers Diamond Oval
Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium
Dubai International Cricket Stadium
Eden Gardens
Green Park
HPCA Stadium
Holkar Cricket Stadium
JSCA International Stadium Complex
Kingsmead
M Chinnaswamy Stadium
MA Chidambaram Stadium
Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur
Maharashtra Cricket Association Stadium
Narendra Modi Stadium
Nehru Stadium
New Wanderers Stadium
Newlands
OUTsurance Oval
PCA Stadium
Rajiv Gandhi International Stadium, Hyderabad
Sardar Patel Stadium, Motera
Saurashtra Cricket Association Stadium
Sawai Mansingh Stadium
Shaheed Veer Narayan Singh International Stadium
Sharjah Cricket Stadium
St George's Park
Subrata Roy Sahara Stadium
SuperSport Park
Vidarbha Cricket Association Stadium, Jamtha
Wankhede Stadium
Zayed Cricket 

In [63]:


matches["venue"] = matches["venue"].replace(venue_mapping)


venues = sorted(matches["venue"].dropna().unique())

print("Total Venues:", len(venues))

for v in venues:
    print(v)

Total Venues: 38
Arun Jaitley Stadium
Barabati Stadium
Barsapara Cricket Stadium, Guwahati
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow
Brabourne Stadium
Buffalo Park
DY Patil Stadium
De Beers Diamond Oval
Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium
Dubai International Cricket Stadium
Eden Gardens
Green Park
HPCA Stadium
Holkar Cricket Stadium
JSCA International Stadium Complex
Kingsmead
M Chinnaswamy Stadium
MA Chidambaram Stadium
Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur
Maharashtra Cricket Association Stadium
Narendra Modi Stadium
Nehru Stadium
New Wanderers Stadium
Newlands
OUTsurance Oval
PCA Stadium
Rajiv Gandhi International Stadium, Hyderabad
Sardar Patel Stadium, Motera
Saurashtra Cricket Association Stadium
Sawai Mansingh Stadium
Shaheed Veer Narayan Singh International Stadium
Sharjah Cricket Stadium
St George's Park
Subrata Roy Sahara Stadium
SuperSport Park
Vidarbha Cricket Association Stadium, Jamtha
Wankhede Stadi

In [64]:
data = matches[
    [
        "team1",
        "team2",
        "venue",
        "toss_winner",
        "toss_decision",
        "winner"
    ]
].dropna()

data.head()

,team1,team2,venue,toss_winner,toss_decision,winner
0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Royal Challengers Bengaluru,field,Kolkata Knight Riders
1,Punjab Kings,Chennai Super Kings,PCA Stadium,Chennai Super Kings,bat,Chennai Super Kings
2,Delhi Capitals,Rajasthan Royals,Arun Jaitley Stadium,Rajasthan Royals,bat,Delhi Capitals
3,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,Mumbai Indians,bat,Royal Challengers Bengaluru
4,Kolkata Knight Riders,Deccan Chargers,Eden Gardens,Deccan Chargers,bat,Kolkata Knight Riders


In [49]:
encoders = {}

for col in data.columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    encoders[col] = le

In [50]:
X = data.drop("winner", axis=1)
y = data["winner"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [51]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, n_estimators=500, random_state=42)

In [54]:
accuracy = accuracy_score(
    y_test,
    model.predict(X_test)
)

print("Accuracy:", round(accuracy*100,2), "%")

Accuracy: 48.62 %


In [67]:
def predict(team1, team2, venue, toss_winner, toss_decision):

    if team1 == team2:
        return "❌ Team 1 and Team 2 cannot be the same."

    row = pd.DataFrame([{
        "team1": encoders["team1"].transform([team1])[0],
        "team2": encoders["team2"].transform([team2])[0],
        "venue": encoders["venue"].transform([venue])[0],
        "toss_winner": encoders["toss_winner"].transform([toss_winner])[0],
        "toss_decision": encoders["toss_decision"].transform([toss_decision])[0]
    }])

    pred = model.predict(row)[0]
    probs = model.predict_proba(row)[0]

    winner = encoders["winner"].inverse_transform([pred])[0]

    confidence = max(probs) * 100

    # Top 3 Predictions
    winner_classes = encoders["winner"].classes_

    top3 = sorted(
        zip(winner_classes, probs),
        key=lambda x: x[1],
        reverse=True
    )[:3]

    top3_text = "\n".join(
        [f"• {team}: {prob*100:.2f}%" for team, prob in top3]
    )

    # Head-to-Head Stats
    h2h = matches[
        (
            (matches["team1"] == team1) &
            (matches["team2"] == team2)
        )
        |
        (
            (matches["team1"] == team2) &
            (matches["team2"] == team1)
        )
    ]

    team1_wins = (h2h["winner"] == team1).sum()
    team2_wins = (h2h["winner"] == team2).sum()

    winner_count = (matches["winner"] == winner).sum()

    summary = f"""
# 🏏 IPL Match Analysis Report

## 🏆 Predicted Winner
### {winner}

---

## 📈 Winning Probability
### {confidence:.2f}%

---

## 📊 Model Performance

✅ Model Accuracy: {accuracy*100:.2f}%

🤖 Algorithm: Random Forest Classifier

🎯 Prediction Type: Multi-Class Classification

---

## 🏟 Match Details

• Team 1: {team1}

• Team 2: {team2}

• Venue: {venue}

• Toss Winner: {toss_winner}

• Toss Decision: {toss_decision}

---

## 🔥 Top Probability Rankings

{top3_text}

---

## ⚔️ Head-to-Head Record

• {team1}: {team1_wins} wins

• {team2}: {team2_wins} wins

• Total Encounters: {len(h2h)}

---

## 📋 Dataset Statistics

• Total Matches: {len(matches)}

• Total Teams: {len(teams)}

• Total Venues: {len(venues)}

• Seasons Covered: {matches['season'].nunique()}

---

## 📌 Historical Insight

• {winner} has won {winner_count} matches in the dataset.

• Prediction uses historical IPL match records.

• Teams, venue and toss information are considered.

---

## ⚠️ Disclaimer

This prediction is based on historical IPL data and machine learning patterns.

Actual results may vary due to player form, injuries, pitch conditions, weather and other real-world factors.
"""

    return summary

In [68]:
teams = sorted(
    set(matches["team1"].dropna())
    .union(set(matches["team2"].dropna()))
)

venues = sorted(matches["venue"].dropna().unique())

In [70]:
with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("""
# 🏏 IPL Match Winner Predictor

### Machine Learning Powered Cricket Analytics Dashboard
""")

    with gr.Row():
        team1 = gr.Dropdown(choices=teams, label="Team 1")
        team2 = gr.Dropdown(choices=teams, label="Team 2")

    venue = gr.Dropdown(choices=venues, label="Venue")

    with gr.Row():
        toss_winner = gr.Dropdown(choices=teams, label="Toss Winner")
        toss_decision = gr.Dropdown(
            choices=["bat", "field"],
            label="Toss Decision"
        )

    btn = gr.Button("🚀 Predict Winner")

    result = gr.Markdown()

    btn.click(
        predict,
        inputs=[
            team1,
            team2,
            venue,
            toss_winner,
            toss_decision
        ],
        outputs=result
    )

app.launch()

/tmp/ipykernel_1710/3917989128.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0b1be998601218cee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
